# AI-alapú részvényelemzés

Ebben a notebookban mesterséges intelligencia eszközök segítségével elemzünk egy részvényt – adatletöltéstől a kereskedési stratégiáig.

### Mit fogunk csinálni?
1. Letöltjük a részvény historikus adatait (+ S&P 500 index referenciaként)
2. Megvizsgáljuk az árfolyamot (USD), a napi hozamot és a kumulált hozamot
3. FinBERT modellel elemezzük a részvényhez kapcsolódó híreket
4. Technikai indikátorok alapján kereskedési stratégiát építünk

### Eszközök
- **yfinance** – részvényadatok letöltése
- **pandas, matplotlib** – adatkezelés és vizualizáció
- **FinBERT** – pénzügyi hírek sentiment elemzése
- **Gemini** – kódgenerálás (vibe coding)

> **Vibe coding:** nem mi írjuk a kódot, hanem AI-jal generáltatjuk. Mi pedig értjük, futtatjuk és irányítjuk.

## 1. Telepítés és importok

Az első lépés a szükséges csomagok telepítése és betöltése. Ezt a cellát mindig futtasd le először!

In [ ]:
# Csomagok telepítése
!pip install yfinance pandas matplotlib -q

# Importok
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
print("Csomagok sikeresen betoltve!")

Csomagok sikeresen betoltve!


## 2. Adatletöltés

Az alábbi cellában beállíthatod a vizsgálni kívánt részvényt és az időszakot.
Az S&P 500 indexet (`^GSPC`) referenciaként mindig letöltjük – összehasonlítási alapként szolgál.

**Paraméterek:**
- `TICKER` – a részvény azonosítója (pl. `AAPL` = Apple, `MSFT` = Microsoft, `TSLA` = Tesla)
- `KEZDO_DATUM` – az elemzés kezdő dátuma (pl. `2020-01-01`)
- `VEGE_DATUM` – az elemzés záró dátuma (pl. `2024-12-31`)

**Adatletöltés logikája:**
1. Ha már van elmentett CSV fájl → azt tölti be
2. Ha nincs → letölti a Yahoo Finance-ről és elmenti
3. Ha a letöltés sem sikerül → betölti a backup adatfájlt

In [ ]:
# ============================================================
# PARAMETEREK – itt tudod modositani!
# ============================================================
TICKER      = "AAPL"       # Reszveny ticker szimbóluma
INDEX       = "^GSPC"      # S&P 500 index (referenciaként)
KEZDO_DATUM = "2020-01-01" # Elemzes kezdo datuma (ÉÉÉÉ-HH-NN)
VEGE_DATUM  = "2024-12-31" # Elemzes zaro datuma  (ÉÉÉÉ-HH-NN)
# ============================================================

def lapitas(adat):
    """MultiIndex oszlopokat lapít – yfinance újabb verziókban szükséges."""
    if isinstance(adat.columns, pd.MultiIndex):
        adat.columns = adat.columns.get_level_values(0)
    return adat

def adatot_betolt(ticker, kezdo, vege):
    """Letölti vagy beolvassa a részvény adatait."""
    csv_fajl = f"{ticker.replace('^','')}_adatok.csv"

    # 1. Már feltöltött/mentett CSV
    if os.path.exists(csv_fajl):
        print(f"Mentett adat betöltése: {csv_fajl}")
        adat = pd.read_csv(csv_fajl, index_col=0, parse_dates=True)
        return lapitas(adat)

    # 2. yfinance letöltés
    try:
        print(f"Letöltés: {ticker} ({kezdo} – {vege})...")
        adat = yf.download(ticker, start=kezdo, end=vege, progress=False)
        if adat.empty:
            raise ValueError("Üres adatsor")
        adat = lapitas(adat)
        adat.to_csv(csv_fajl)
        print(f"Letöltve és elmentve: {len(adat)} sor")
        return adat
    except Exception as e:
        print(f"Letöltés sikertelen ({e})")
        print(f"Töltsd fel manuálisan: {csv_fajl}")
        print("Bal oldali fájlkezelő → feltöltés → futtasd újra a cellát")
        return None

# Részvény és index letöltése
df       = adatot_betolt(TICKER, KEZDO_DATUM, VEGE_DATUM)
df_index = adatot_betolt(INDEX,  KEZDO_DATUM, VEGE_DATUM)

# Előnézet
if df is not None:
    print(f"\nRészvény adatok ({TICKER}): {len(df)} sor")
    display(df.tail(3))
if df_index is not None:
    print(f"\nIndex adatok ({INDEX}): {len(df_index)} sor")

Letöltés: AAPL (2020-01-01 – 2024-12-31)...


/tmp/ipykernel_304/281429866.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  adat = yf.download(ticker, start=kezdo, end=vege, progress=False)


Letöltve és elmentve: 1257 sor
Letöltés: ^GSPC (2020-01-01 – 2024-12-31)...


/tmp/ipykernel_304/281429866.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  adat = yf.download(ticker, start=kezdo, end=vege, progress=False)


Letöltve és elmentve: 1257 sor

Részvény adatok (AAPL): 1257 sor


Price,Close,High,Low,Open,Volume
Date,,,,,
2024-12-26,257.612701,258.686851,256.230269,256.787224,27237100
2024-12-27,254.201370,257.294489,251.685117,256.429191,42355300
2024-12-30,250.829803,252.122744,249.387684,250.859639,35557500



Index adatok (^GSPC): 1257 sor


## 3. Árfolyam vizualizáció

Az első vizualizáció: hogyan alakult a részvény záróárfolyama USD-ben az elmúlt években?
A `df` DataFrame `Close` oszlopa tartalmazza a napi záróárakat.

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely megjeleníti a részvény árfolyamát!

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [1]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Árfolyam vizualizáció – részvény záróára USD-ben


**Módosító prompt feladat**

Ha megvan az alap diagram, kérd meg a Geminit, hogy jelölje meg a fontos időszakokat:
- COVID-összeomlás: 2020-02 – 2020-06
- Ukrajna háború + kamatemelés: 2022-02 – 2022-12

In [2]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Módosítás: krízis időszakok kiemelése



## 4. Napi hozam

A napi hozam (daily return) megmutatja, mekkora volt az árfolyam napi változása százalékban:

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}} \times 100$$

- Pozitív érték → aznap emelkedett az árfolyam
- Negatív érték → aznap esett
- Nagy kilengések → magas volatilitás, kockázatos időszak

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára a napi hozam kiszámításához és megjelenítéséhez!
Csak a részvény (`df`) napi hozamát jelenítsd meg.

*A kapott kódot másold be az alábbi cellába és futtasd le!*

---
**Módosító prompt feladat**

Ha megvan az alap diagram, kérd meg a Geminit, hogy emelje ki:
- A legjobb és legrosszabb napot (pont + felirat a dátummal és értékkel)
- Egy vízszintes vonalat a napi hozamok átlagánál

In [3]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Napi hozam vizualizáció



In [ ]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Módosítás: legjobb/legrosszabb nap + átlagvonal


## 5. Kumulált hozam

A kumulált hozam megmutatja: ha 2020 elején 1000 dollárt fektettél volna be, ma mennyi lenne?
Ezt a részvényre és az S&P 500 indexre egyszerre vizualizáljuk – így látszik, túlteljesítette-e a részvény a piacot.

- A vonal meredeksége → milyen gyorsan nőtt a befektetés értéke
- Ha a részvény vonala az index felett van → a részvény jobban teljesített a piacnál
- Ha alatta → érdemes lett volna inkább indexalapba fektetni

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely kiszámítja és megjeleníti mindkét eszköz kumulált hozamát!
A `df` a részvény, a `df_index` az S&P 500 adatait tartalmazza.

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [4]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Kumulált hozam – részvény vs. S&P 500



---
## 6. FinBERT sentiment elemzés

A **FinBERT** egy pénzügyi szövegekre finomhangolt nyelvi modell – kifejezetten részvénypiaci hírek,
elemzések és jelentések értelmezésére tanították be.
Minden hírhez három értéket ad:

- **positive** – pozitív hangulat valószínűsége
- **negative** – negatív hangulat valószínűsége
- **neutral** – semleges hangulat valószínűsége

A három érték összege mindig 1.

### 6.1 Telepítés és modell betöltése

> **Figyelem:** A FinBERT modell (~400 MB) az első futtatáskor letöltodik a Hugging Face szervereirol.
> Ez 1-2 percet vehet igénybe – ez normális, csak egyszer kell letölteni!

In [ ]:
# FinBERT telepítése
!pip install transformers torch -q

from transformers import BertTokenizer, BertForSequenceClassification
import torch
import pandas as pd

# Modell betöltése – elso futáskor letölti (~400 MB)
print("FinBERT modell betöltése... (elso futáskor 1-2 perc)")
MODEL_NEV = "ProsusAI/finbert"
tokenizer = BertTokenizer.from_pretrained(MODEL_NEV)
model     = BertForSequenceClassification.from_pretrained(MODEL_NEV)
model.eval()
print("Modell betöltve!")

FinBERT modell betöltése... (elso futáskor 1-2 perc)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modell betöltve!


### 6.2 Hírek betöltése

Az `AAPL_hirek.csv` fájlt töltsd fel a Colab session-be (bal oldali fájlkezelő → feltöltés).
A fájl 55 ellenőrzött Apple-hírt tartalmaz 2020-2024 közötti időszakból.

In [ ]:
# Hírek betöltése
hirek = pd.read_csv("AAPL_hirek.csv", parse_dates=["date"])
hirek = hirek.sort_values("date").reset_index(drop=True)
print(f"Betöltött hírek: {len(hirek)} db")
print(f"Időszak: {hirek['date'].min().date()} – {hirek['date'].max().date()}")
display(hirek.head())

Betöltött hírek: 55 db
Időszak: 2020-01-28 – 2024-11-14


,date,headline
0,2020-01-28,Apple Reports Record Q1 FY2020 Revenue of $91....
1,2020-02-17,Apple Issues Revenue Warning Withdrawing Q2 Gu...
2,2020-03-13,Apple Closes All Retail Stores Outside Greater...
3,2020-03-13,Apple Announces $15 Million in COVID-19 Donati...
4,2020-04-10,Apple and Google Announce Joint COVID-19 Conta...


### 6.3 Sentiment elemzés futtatása

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely minden hírhez kiszámítja a FinBERT sentiment score-t!

Kontextus:
- `hirek` egy pandas DataFrame, `date` és `headline` oszlopokkal
- `tokenizer` és `model` már be van töltve
- Az eredményt add hozzá a `hirek` DataFrame-hez: `sentiment` (positive/negative/neutral) és `score` oszlopok

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [5]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# FinBERT sentiment elemzés – minden hírre



### 6.4 Eredmények táblázatban

---
**Prompt feladat**

Az előző kód folytatásaként jelenítsd meg az eredményeket egy áttekinthető táblázatban!
Legyen látható a dátum, a hír címe, a sentiment (positive/negative/neutral) és a score.

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [6]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Sentiment eredmények táblázatban



### 6.5 Top 5 legpozitívabb és legnegatívabb hír

Melyik hír kapta a legmagasabb pozitív score-t? Melyik a legnegatívabbat?

---
**Prompt feladat**

Az előző kód folytatásaként jelenítsd meg a top 5 legpozitívabb és
top 5 legnegatívabb hírt két külön táblázatban, score szerint rendezve!

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [7]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Top 5 legpozitívabb és legnegatívabb hír



### 6.6 Sentiment megoszlás

Összességében hogyan oszlanak meg a hírek?

---
**Prompt feladat**

Az előző kód folytatásaként készíts kördiagramot a sentiment kategóriák megoszlásáról!

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [8]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Sentiment megoszlás kördiagram



### 6.7 Hírek az árfolyamon

Az utolsó vizualizáció: minden hír megjelenik az árfolyam diagramon egy pontként –
zölddel a pozitív, pirossal a negatív híreket jelölve.

> **Megjegyzés:** 55 hírből statisztikai korrelációt számolni nem lenne megbízható.
> Amit viszont látunk: a FinBERT helyesen azonosítja a negatív és pozitív híreket –
> ez önmagában is értékes eredmény. Megbízható korreláció számításhoz
> több száz hír kellene egyenletes időbeli elosztással.

---
**Prompt feladat**

Az előző kód folytatásaként jelenítsd meg az AAPL árfolyamát (`df`),
és rajzold rá a híreket pontként: zöld = pozitív, piros = negatív, szürke = semleges.
Legyen jelmagyarázat és cím.

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [9]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Árfolyam + hírek scatter plot (zöld=pozitív, piros=negatív, szürke=semleges)



---
## 7. Kereskedési stratégia

Az eddig látott árfolyam és sentiment adatok alapján most egy egyszerű
**kereskedési stratégiát** építünk – mozgóátlagok alapján.

Az ötlet egyszerű: ha a rövid távú átlag átlép a hosszú távú átlag fölé,
az emelkedő trendet jelzi → vásárlási jel. Ha alá kerül → eladási jel.

Ezt a két keresztezési pontot hívják:
- **Golden cross** – rövid átlag átlép a hosszú fölé → vétel
- **Death cross** – rövid átlag átlép a hosszú alá → eladás

### 7.1 Technikai indikátorok (előre megírva)

Kiszámítjuk az 50 és 200 napos mozgóátlagot, és megjelenítjük az árfolyamon.

In [10]:
# Mozgóátlagok kiszámítása
df['MA50']  = df['Close'].rolling(window=50).mean()
df['MA200'] = df['Close'].rolling(window=200).mean()

# Vizualizáció
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], label='Záróárfolyam', color='royalblue', linewidth=1.2, alpha=0.7)
plt.plot(df.index, df['MA50'],  label='50 napos mozgóátlag', color='orange', linewidth=1.5)
plt.plot(df.index, df['MA200'], label='200 napos mozgóátlag', color='red',    linewidth=1.5)
plt.title('AAPL árfolyam és mozgóátlagok', fontsize=14)
plt.xlabel('Dátum')
plt.ylabel('Árfolyam (USD)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()
print('Mozgóátlagok kiszámítva: MA50 és MA200 hozzáadva a df-hez')

NameError: name 'df' is not defined

### 7.2 Kereskedési jelzések generálása

Most meghatározzuk a vételi és eladási jelzések pontjait:
- **Golden cross:** az MA50 átlép az MA200 fölé → vétel
- **Death cross:** az MA50 átlép az MA200 alá → eladás

---
**Prompt feladat**

Fogalmazz meg egy promptot a Gemini számára, amely meghatározza
a vételi és eladási jelzések pontjait, és megjeleníti őket az árfolyamon!

Kontextus:
- `df` DataFrame tartalmazza a `Close`, `MA50`, `MA200` oszlopokat
- Golden cross: MA50 átlép MA200 fölé → zöld háromszög jelölje
- Death cross: MA50 átlép MA200 alá → piros háromszög jelölje

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [11]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Kereskedési jelzések – golden cross és death cross



### 7.3 Backtesting – hogyan teljesített a stratégia?

Ha 2020 elején 1000 dollárral indultunk és minden jelzésnél vettünk/eladtunk,
mennyit ér most a portfóliónk? Megverte-e a stratégia az egyszerű buy & hold megközelítést?

---
**Prompt feladat**

Az előző kód folytatásaként szimulálj egy egyszerű backtestet!
1000 dolláros kezdőtőkével, minden golden cross-nál vétel, minden death cross-nál eladás.
Számítsd ki a végső portfólió értékét.

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [11]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Backtesting


### 7.4 Stratégia vs. Buy & Hold

Az utolsó vizualizáció: hogyan alakult a portfólió értéke időben –
a mozgóátlag stratégiával és az egyszerű buy & hold megközelítéssel összehasonlítva.

---
**Prompt feladat**

Az előző kód folytatásaként jelenítsd meg egy diagramon a mozgóátlag stratégia
és a buy & hold portfólió értékét időben, 1000 dolláros kezdőtőkéről indulva.
Legyen cím, tengelyfeliratok és jelmagyarázat.

*A kapott kódot másold be az alábbi cellába és futtasd le!*

In [12]:
# IDE KERÜL A GEMINI ÁLTAL GENERÁLT KÓD
# Stratégia vs. Buy & Hold összehasonlítás

